# NB01 · Configuration & Setup
**Run this notebook first, or `%run ./NB01_Config_and_Setup` at the top of every other notebook.**

## 1 · Table Path Variables
> **Edit only this cell** to point to your Databricks tables or file paths.

In [0]:

BILLINGS_TABLE      = "workspace.default.billings_raw"    
CC_CALLS_TABLE      = "workspace.default.cc_calls"        
EMAILS_TABLE        = "workspace.default.emails"
RENEWAL_CALLS_TABLE = "workspace.default.renewal_calls"   



CHURN_DAYS_THRESHOLD = 28     
TRAINING_YEAR        = 2023  
TARGET_YEARS         = [2023, 2024, 2025]


print("[NB01] Config loaded")
print(f"  Billings : {BILLINGS_TABLE}")
print(f"  CC Calls : {CC_CALLS_TABLE}")
print(f"  Emails   : {EMAILS_TABLE}")
print(f"  Renewals : {RENEWAL_CALLS_TABLE}")
print(f"  Churn threshold: {CHURN_DAYS_THRESHOLD} days")


## 2 · Imports

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor':'#0a0e1a','axes.facecolor':'#111827',
    'axes.edgecolor':'#1e2d45','axes.labelcolor':'#94a3b8',
    'axes.titlecolor':'#e2e8f0','xtick.color':'#64748b','ytick.color':'#64748b',
    'text.color':'#e2e8f0','grid.color':'#1e2d45','grid.alpha':0.5,
    'figure.titlesize':15,'axes.titlesize':12,'axes.labelsize':10,
    'legend.facecolor':'#1a2236','legend.edgecolor':'#1e2d45',
})
PALETTE = {'danger':'#ef4444','amber':'#f59e0b','blue':'#3b82f6','green':'#10b981','muted':'#334155'}

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, pointbiserialr
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report, roc_curve, confusion_matrix
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("[NB01] Libraries imported")


## 3 · Shared Helper Functions

In [0]:
def load_table(table_name, path_fallback=None):
    """Load a Databricks Delta table or fall back to CSV."""
    try:
        df = spark.table(table_name).toPandas()
        print(f"  [{table_name}] {df.shape[0]:,} rows x {df.shape[1]} cols")
    except Exception:
        if path_fallback:
            df = pd.read_csv(path_fallback, low_memory=False)
            print(f"  [CSV fallback: {path_fallback}] {df.shape[0]:,} rows x {df.shape[1]} cols")
        else:
            raise
    return df

def parse_dates(df, cols, dayfirst=True):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], dayfirst=dayfirst, errors='coerce')
    return df

def binary_flags(df, cols, yes_val='Yes'):
    for c in cols:
        if c in df.columns:
            df[c] = (df[c] == yes_val).astype(int)
    return df

def section(title):
    print(f"\n{'='*65}\n  {title}\n{'='*65}")

def pct(n, d):
    return round(n/d*100, 2) if d else 0

def display_df(df, title=None, n=20):
    if title: print(f"\n>>> {title}")
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def year_color(year):
    return {2023:PALETTE['danger'],2024:PALETTE['amber'],2025:PALETTE['green']}.get(year, PALETTE['blue'])

def rate_color(rate):
    if rate >= 12: return PALETTE['danger']
    if rate >= 9:  return PALETTE['amber']
    return PALETTE['green']

print("[NB01] Helper functions registered")
